# 15 - Post-processing with Linguistic Rules

**Technique:** Rule-based refinement of model predictions  
**Approach:**
1. Load best model/ensemble predictions
2. Apply Arabic linguistic rules for common diacritization patterns
3. Fix common errors (shadda combinations, tanween placement, etc.)

**Base Options (choose one):**
- `ensemble_voting` - From notebook 13
- `ensemble_weighted` - From notebook 14  
- `catt` / `arabert` / `tarteel_whisper` - Individual best models (0.10 WER)

**Tasks:**
1. Dev Set: Load predictions + Apply rules + Metrics
2. Blind Test: Load predictions + Apply rules + Submission JSON

In [1]:
!pip install -q tqdm pyarabic


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Setup - Import from config.py (generated by setup.sh)
import os, sys, json, re, zipfile
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

# Add project root to path for config import
IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

# Import paths from config
try:
    from config import (
        PROJECT_DIR, DATA_DIR,
        DEV_OUTPUT,
        OUTPUT_DIR, SUBMISSION_DIR
    )
except ImportError:
    print("WARNING: config.py not found. Run setup.sh first!")
    DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
    DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
    OUTPUT_DIR = PROJECT_DIR / 'outputs'
    SUBMISSION_DIR = PROJECT_DIR / 'submissions'

MODEL_KEY = 'postprocess'

# Choose base model to post-process:
# - 'ensemble_voting' or 'ensemble_weighted' (run notebook 13/14 first)
# - 'catt', 'arabert', 'tarteel_whisper' etc. (individual models with 0.10 WER)
BASE_MODEL = 'ensemble_voting'  # Change this to use different base

print(f"Environment: 'Local'")
print(f"Base model: {BASE_MODEL}")

Environment: Local
Base model: ensemble_voting


In [3]:
# Arabic diacritics
FATHA = '\u064E'      # َ
DAMMA = '\u064F'      # ُ
KASRA = '\u0650'      # ِ
SHADDA = '\u0651'     # ّ
SUKUN = '\u0652'      # ْ
FATHATAN = '\u064B'   # ً
DAMMATAN = '\u064C'   # ٌ
KASRATAN = '\u064D'   # ٍ

DIACRITICS = {FATHA, DAMMA, KASRA, SHADDA, SUKUN, FATHATAN, DAMMATAN, KASRATAN}
HARAKAT = {FATHA, DAMMA, KASRA, SUKUN, FATHATAN, DAMMATAN, KASRATAN}
TANWEEN = {FATHATAN, DAMMATAN, KASRATAN}

# Common Arabic letters
ALEF = '\u0627'       # ا
ALEF_MADDA = '\u0622' # آ
ALEF_HAMZA_ABOVE = '\u0623'  # أ
ALEF_HAMZA_BELOW = '\u0625'  # إ
WAW = '\u0648'        # و
YA = '\u064A'         # ي
TA_MARBUTA = '\u0629' # ة
HAMZA = '\u0621'      # ء
LAM = '\u0644'        # ل

DIACRITIC_PATTERN = re.compile(r'[\u064B-\u0652]')
ARABIC_LETTER_PATTERN = re.compile(r'[\u0621-\u064A]')
IRAB_DIACRITICS = {'\u064B', '\u064C', '\u064D'}

In [4]:
def normalize_diacritics(text):
    """
    Normalize diacritic ordering and remove duplicates.
    Shadda should come before other diacritics.
    """
    result = []
    i = 0
    
    while i < len(text):
        char = text[i]
        
        if char in DIACRITICS:
            i += 1
            continue
        
        result.append(char)
        
        # Collect following diacritics
        diacs = []
        j = i + 1
        while j < len(text) and text[j] in DIACRITICS:
            if text[j] not in diacs:  # Remove duplicates
                diacs.append(text[j])
            j += 1
        
        # Normalize order: shadda first
        if SHADDA in diacs:
            diacs.remove(SHADDA)
            diacs.insert(0, SHADDA)
        
        # Only keep one haraka (last one wins, after shadda)
        harakat_found = [d for d in diacs if d in HARAKAT]
        if len(harakat_found) > 1:
            # Keep shadda + last haraka
            diacs = [d for d in diacs if d == SHADDA] + [harakat_found[-1]]
        
        result.extend(diacs)
        i = j
    
    return ''.join(result)

In [5]:
def fix_alef_rules(text):
    """
    Fix common Alef diacritization rules.
    - Alef at word start often has hamza
    - Alef shouldn't have sukun
    """
    # Remove sukun from alef
    text = re.sub(f'[{ALEF}{ALEF_MADDA}{ALEF_HAMZA_ABOVE}{ALEF_HAMZA_BELOW}]{SUKUN}', 
                  lambda m: m.group(0)[0], text)
    
    return text

def fix_tanween_placement(text):
    """
    Fix tanween placement.
    - Fathatan should be on alef before ta marbuta: ة -> ةً or on alef
    - Tanween at end of word only
    """
    # Move fathatan from ta marbuta to preceding alef if exists
    text = re.sub(f'{ALEF}({TA_MARBUTA}){FATHATAN}', 
                  f'{ALEF}{FATHATAN}\\1', text)
    
    return text

def fix_lam_alef(text):
    """
    Fix Lam-Alef (definite article) diacritization.
    - الـ usually has sukun on lam or shadda on following letter (solar)
    """
    # Common pattern: ال followed by solar letter should have shadda
    SOLAR_LETTERS = 'تثدذرزسشصضطظلن'
    
    for solar in SOLAR_LETTERS:
        # Add shadda to solar letter after ال if missing
        pattern = f'{ALEF}{LAM}([{FATHA}{KASRA}]?)({solar})(?![{SHADDA}])'
        text = re.sub(pattern, f'{ALEF}{LAM}\\1\\2{SHADDA}', text)
    
    return text

In [6]:
def apply_postprocessing(text):
    """
    Apply all post-processing rules.
    """
    # 1. Normalize diacritics
    text = normalize_diacritics(text)
    
    # 2. Fix Alef rules
    text = fix_alef_rules(text)
    
    # 3. Fix tanween placement
    text = fix_tanween_placement(text)
    
    # 4. Fix Lam-Alef (be careful - can over-correct)
    # text = fix_lam_alef(text)  # Disabled by default
    
    return text

In [7]:
# Load predictions
def load_predictions(model_key, pred_type='dev'):
    pred_file = OUTPUT_DIR / f"{model_key}_{pred_type}_predictions.json"
    if pred_file.exists():
        with open(pred_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

In [8]:
# Metrics
def extract_diacritics(text):
    result = []
    i = 0
    while i < len(text):
        if ARABIC_LETTER_PATTERN.match(text[i]):
            diacritics = []
            j = i + 1
            while j < len(text) and DIACRITIC_PATTERN.match(text[j]):
                diacritics.append(text[j])
                j += 1
            result.append((text[i], ''.join(diacritics)))
            i = j
        else:
            i += 1
    return result

def compute_metrics(predictions, ground_truth, exclude_irab=False):
    gt_lookup = {item['id']: item['text_diacritized'] for item in ground_truth}
    total_chars, total_words, char_errors, word_errors, ser_sum, n = 0, 0, 0, 0, 0, 0
    for pred in predictions:
        if pred['id'] not in gt_lookup: continue
        pred_text, ref_text = pred['text_diacritized'], gt_lookup[pred['id']]
        pred_pairs, ref_pairs = extract_diacritics(pred_text), extract_diacritics(ref_text)
        for i in range(min(len(pred_pairs), len(ref_pairs))):
            pred_d, ref_d = pred_pairs[i][1], ref_pairs[i][1]
            if exclude_irab:
                pred_d = ''.join(d for d in pred_d if d not in IRAB_DIACRITICS)
                ref_d = ''.join(d for d in ref_d if d not in IRAB_DIACRITICS)
            if pred_d != ref_d: char_errors += 1
        char_errors += abs(len(pred_pairs) - len(ref_pairs))
        total_chars += max(len(pred_pairs), len(ref_pairs))
        pred_words, ref_words = pred_text.split(), ref_text.split()
        for i in range(min(len(pred_words), len(ref_words))):
            if pred_words[i] != ref_words[i]: word_errors += 1
        word_errors += abs(len(pred_words) - len(ref_words))
        total_words += max(len(pred_words), len(ref_words))
        if pred_text != ref_text: ser_sum += 1
        n += 1
    return {'DER': char_errors/total_chars if total_chars else 0, 'WER': word_errors/total_words if total_words else 0, 'SER': ser_sum/n if n else 0, 'n_samples': n}

## Dev Set Post-processing

In [9]:
dev_preds = load_predictions(BASE_MODEL, 'dev')
print(f"Loaded {len(dev_preds)} dev predictions from {BASE_MODEL}")

# Apply post-processing
dev_postprocessed = []
for pred in tqdm(dev_preds, desc="Post-processing dev"):
    processed_text = apply_postprocessing(pred['text_diacritized'])
    dev_postprocessed.append({
        'id': pred['id'],
        'text_diacritized': processed_text
    })

Loaded 260 dev predictions from ensemble_voting


Post-processing dev: 100%|██████████| 260/260 [00:00<00:00, 46173.22it/s]


In [10]:
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

print("="*60)
print(f"BEFORE POST-PROCESSING ({BASE_MODEL})")
print("="*60)
m_before = compute_metrics(dev_preds, dev_output, False)
print(f"[With I'rab] WER: {m_before['WER']*100:.2f}%")

print("\n" + "="*60)
print("AFTER POST-PROCESSING")
print("="*60)
m1 = compute_metrics(dev_postprocessed, dev_output, False)
print(f"\n[With I'rab] DER: {m1['DER']*100:.2f}% | WER: {m1['WER']*100:.2f}% (PRIMARY) | SER: {m1['SER']*100:.2f}%")
m2 = compute_metrics(dev_postprocessed, dev_output, True)
print(f"[No I'rab]   DER: {m2['DER']*100:.2f}% | WER: {m2['WER']*100:.2f}% | SER: {m2['SER']*100:.2f}%")

improvement = (m_before['WER'] - m1['WER']) * 100
print(f"\nWER improvement: {improvement:+.2f}%")

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_postprocessed, f, ensure_ascii=False, indent=2)

BEFORE POST-PROCESSING (ensemble_voting)
[With I'rab] WER: 40.87%

AFTER POST-PROCESSING

[With I'rab] DER: 13.29% | WER: 41.09% (PRIMARY) | SER: 79.23%
[No I'rab]   DER: 13.22% | WER: 41.09% | SER: 79.23%

WER improvement: -0.22%


## Blind Test Post-processing

In [11]:
test_preds = load_predictions(BASE_MODEL, 'test')
print(f"Loaded {len(test_preds)} test predictions from {BASE_MODEL}")

test_postprocessed = []
for pred in tqdm(test_preds, desc="Post-processing test"):
    processed_text = apply_postprocessing(pred['text_diacritized'])
    test_postprocessed.append({
        'id': pred['id'],
        'text_diacritized': processed_text
    })

Loaded 328 test predictions from ensemble_voting


Post-processing test: 100%|██████████| 328/328 [00:00<00:00, 47250.02it/s]


In [12]:
with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_postprocessed, f, ensure_ascii=False, indent=2)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_postprocessed, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print(f"\nSUBMISSION READY: {zip_file} ({zip_file.stat().st_size/1024:.1f} KB)")


SUBMISSION READY: ./submissions/postprocess_submission_20260220_152557.zip (18.4 KB)
